# PicMap v2 — Phase 1: Extract Metadata + Sample Photos

This notebook extracts GPS/timestamp metadata from your Takeout album and fetches 3 sample photos per stop.

**Run this once** to generate:
- `trip_metadata.json` — All photo GPS/timestamps for local iteration
- `data.json` — Stop/route data for the frontend
- `output/photos/` — ~100-300 sample photos (3 per stop)

**Phase 2 (local):** Edit `config.json` clustering thresholds and run `build_from_metadata.py` locally — no Drive needed.

In [1]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── Cell 2: Clone / pull latest code from GitHub ────────────────────────
import os

REPO_URL = 'https://github.com/cmprice1/picmap.git'
BRANCH   = 'v2-colab'
REPO_DIR = '/content/picmap'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    print('Cloning repo...')
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

!echo "Latest commit: $(git -C {REPO_DIR} log -1 --format='%h %s')"

Cloning repo...
Cloning into '/content/picmap'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 162 (delta 20), reused 33 (delta 12), pack-reused 113 (from 1)
Receiving objects: 100% (162/162), 169.35 KiB | 2.73 MiB/s, done.
Resolving deltas: 100% (77/77), done.
Latest commit: 0e47ee6 Redesign pipeline: Phase 1 (Colab) + Phase 2 (local)


In [3]:
# ── Cell 3: Install dependencies ────────────────────────────────────────
!pip install -q -r {REPO_DIR}/requirements.txt
print('Dependencies ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 47.3 MB/s eta 0:00:00a 0:00:01
Dependencies ready.


In [4]:
# ── Cell 4: Run Phase 1 — Extract metadata + sample photos ──────────────
# This:
#   1. Copies just .json sidecars to local (fast, ~24MB)
#   2. Parses all GPS/timestamps
#   3. Clusters with loose thresholds (more stops = better coverage)
#   4. Fetches 3 sample photos per stop (~100-300 photos total)
#   5. Outputs trip_metadata.json + data.json

ALBUM = '/content/drive/My Drive/PicMap-V2 Project/Takeout/Google Photos/2025 Great American Roadtrip (iphone)'
OUTPUT = f'{REPO_DIR}/output'
CONFIG = f'{REPO_DIR}/config_phase1.json'
LOCAL_CACHE = '/content/local_sidecars'

!python {REPO_DIR}/extract_metadata.py \
    --album "{ALBUM}" \
    --output "{OUTPUT}" \
    --config "{CONFIG}" \
    --local-cache "{LOCAL_CACHE}"

Config: cluster_radius=1.5km, time_gap=1.5h, samples_per_stop=3
Copying sidecars from: 2025 Great American Roadtrip (iphone)
  Found 8435 sidecar files
  ...copied 100/8435 sidecars
  ...copied 200/8435 sidecars
  ...copied 300/8435 sidecars
  ...copied 400/8435 sidecars
  ...copied 500/8435 sidecars
  ...copied 600/8435 sidecars
  ...copied 700/8435 sidecars
  ...copied 800/8435 sidecars
  ...copied 900/8435 sidecars
  ...copied 1000/8435 sidecars
  ...copied 1100/8435 sidecars
  ...copied 1200/8435 sidecars
  ...copied 1300/8435 sidecars
  ...copied 1400/8435 sidecars
  ...copied 1500/8435 sidecars
  ...copied 1600/8435 sidecars
  ...copied 1700/8435 sidecars
  ...copied 1800/8435 sidecars
  ...copied 1900/8435 sidecars
  ...copied 2000/8435 sidecars
  ...copied 2100/8435 sidecars
  ...copied 2200/8435 sidecars
  ...copied 2300/8435 sidecars
  ...copied 2400/8435 sidecars
  ...copied 2500/8435 sidecars
  ...copied 2600/8435 sidecars
  ...copied 2700/8435 sidecars
  ...copied 2800/843

In [5]:
# ── Cell 5: Save outputs to Drive (persistent backup) ──────────────────────
import shutil
import os

# Create backup folder in Drive
BACKUP_DIR = '/content/drive/My Drive/PicMap-V2 Project/Phase1-Outputs'
os.makedirs(BACKUP_DIR, exist_ok=True)

# Copy small metadata files to Drive
small_files = ['output/trip_metadata.json', 'output/data.json']
for fname in small_files:
    src = f'{REPO_DIR}/{fname}'
    dst = f'{BACKUP_DIR}/{os.path.basename(fname)}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        file_size = os.path.getsize(dst) / 1024 / 1024
        print(f'✓ Saved {os.path.basename(fname)} ({file_size:.1f}MB) → Drive')

# Also copy sample photos folder
photos_src = f'{REPO_DIR}/output/photos'
photos_dst = f'{BACKUP_DIR}/photos'
if os.path.exists(photos_src) and os.path.isdir(photos_src):
    if os.path.exists(photos_dst):
        shutil.rmtree(photos_dst)
    shutil.copytree(photos_src, photos_dst)
    num_photos = len([f for f in os.listdir(photos_dst) if os.path.isfile(os.path.join(photos_dst, f))])
    print(f'✓ Saved {num_photos} sample photos → Drive')

print(f'\n✓ All backups saved to: {BACKUP_DIR}')
print('\nTo sync to GitHub locally:')
print('  1. Download trip_metadata.json & data.json from Drive')
print('  2. Save to your local roadtrip-map/output/ folder')
print('  3. git add output/trip_metadata.json output/data.json')
print('  4. git commit -m "Phase 1: Extract metadata + sample photos"')
print('  5. git push origin v2-colab')

✓ Saved trip_metadata.json (1.4MB) → Drive
✓ Saved data.json (0.4MB) → Drive
✓ Saved 1010 sample photos → Drive

✓ All backups saved to: /content/drive/My Drive/PicMap-V2 Project/Phase1-Outputs

To sync to GitHub locally:
  1. Download trip_metadata.json & data.json from Drive
  2. Save to your local roadtrip-map/output/ folder
  3. git add output/trip_metadata.json output/data.json
  4. git commit -m "Phase 1: Extract metadata + sample photos"
  5. git push origin v2-colab


In [6]:
# ── Cell 6: Preview results ─────────────────────────────────────────────
import json
import os

with open(f'{REPO_DIR}/output/data.json') as f:
    data = json.load(f)

with open(f'{REPO_DIR}/output/trip_metadata.json') as f:
    metadata = json.load(f)

photos_dir = f'{REPO_DIR}/output/photos'
num_photos = len([f for f in os.listdir(photos_dir) if os.path.isfile(os.path.join(photos_dir, f))])

print(f"✓ Trip: {data['trip']['title']}")
print(f"✓ Date range: {data['trip']['start_date']} to {data['trip']['end_date']}")
print(f"\n✓ Metadata: {len(metadata['photos'])} total photos with GPS/timestamps")
print(f"✓ Sample photos downloaded: {num_photos}")
print(f"✓ Stops: {len(data['stops'])}")
print(f"✓ Waypoints: {len(data['waypoints'])}")
print(f"\n{'='*70}")
print("NEXT: Download files from Drive and commit locally")
print(f"{'='*70}\n")
print(f"Files saved to Drive: /PicMap-V2 Project/Phase1-Outputs/\n")
print("Stops breakdown:")
for s in data['stops'][:10]:  # Show first 10
    marker = '●' if s['type'] == 'overnight' else '○'
    print(f"  {marker} {s['order']:2d}. {s['name'][:28]:<28} ({s['photo_count']:4d} photos)")
if len(data['stops']) > 10:
    print(f"  ... and {len(data['stops']) - 10} more stops")

✓ Trip: Great American Road Trip
✓ Date range: 1970-01-01 to 2025-08-03

✓ Metadata: 8435 total photos with GPS/timestamps
✓ Sample photos downloaded: 1010
✓ Stops: 154
✓ Waypoints: 245

NEXT: Download files from Drive and commit locally

Files saved to Drive: /PicMap-V2 Project/Phase1-Outputs/

Stops breakdown:
  ●  1. Los Angeles                  (  13 photos)
  ○  2. Los Angeles                  (  28 photos)
  ○  3. Carpinteria                  ( 816 photos)
  ○  4. Santa Barbara                (  25 photos)
  ○  5. Santa Barbara                (  30 photos)
  ○  6. Santa Barbara                (  52 photos)
  ○  7. Santa Barbara                (  13 photos)
  ○  8. Santa Barbara                (  12 photos)
  ○  9. Santa Barbara County         (  51 photos)
  ○ 10. Santa Barbara County         (  26 photos)
  ... and 144 more stops


---

## Next Steps (Phase 2 — Local)

1. **Pull to your local machine:**
   ```bash
   git pull origin v2-colab
   ```

2. **Iterate on clustering locally:**
   - Edit `config.json` thresholds (`cluster_radius_km`, `cluster_time_gap_hours`)
   - Run: `python build_from_metadata.py --config config.json`
   - Test: `python -m http.server 8080 --directory output`

3. **When happy with stops, deploy to Netlify**